In [1]:
#1. Retrieval: gives results from vector store based on query
# def retrieve: Retrieve relevant documents from the vector store in form of a list of dictionaries 
# query is tranformed into an embedding and then used to search the vector store for similar documents

import sys
sys.path.insert(0, '../')

from typing import List, Dict, Any
from src.embedding_manager import EmbeddingManager
from src.vector_store import FaissVectorStore

class RAGRetriever:
    def __init__(self, vector_store: FaissVectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        
        try:
            # Generate embedding for the query
            query_embeddings = self.embedding_manager.generate_embeddings([query])
            query_embedding = query_embeddings[0]
            
            # Search the vector store
            results = self.vector_store.search(query_embedding, top_k=top_k)
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadata = results['metadata'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, meta, distance) in enumerate(zip(ids, metadata, distances)):
                    similarity_score = 1 - distance 

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id, 
                            'content': meta.get('content', ''),
                            'source': meta.get('source', ''),
                            'similarity_score': float(similarity_score),
                            'metadata': meta,
                            'rank': i + 1
                        })
                
                if retrieved_docs:
                    print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}")
                else:
                    print(f"No documents retrieved above the score threshold of {score_threshold}")
            else:
                print("No results found in vector store")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        


/Users/merlindconstanzepohl/Desktop/RAG Implementierung/Code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#2. Initialize vector store and embedding manager (load from disk)

try:
    embedding_manager = EmbeddingManager()
    print("EmbeddingManager initialized successfully")
except Exception as e:
    print(f"Failed to initialize EmbeddingManager: {e}")
    embedding_manager = None

try:
    vector_store = FaissVectorStore(embedding_dim=embedding_manager.get_embedding_dimension())
    print(f"Vector store loaded successfully with {len(vector_store.id_to_metadata)} chunks")
except Exception as e:
    print(f"Failed to initialize Vector Store: {e}")
    vector_store = None

In [ ]:
# 3. Initialize RAGRetriever with instances from ingestion.ipynb

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"
results = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

for doc in results:
    print(f"\nRank {doc['rank']}: {doc['similarity_score']:.4f}")
    print(f"Source: {doc['source']}")
    print(f"Content preview: {doc['content'][:200]}...")


NameError: name 'vector_store' is not defined